<a href="https://colab.research.google.com/github/Danodia-Rahul/Directors-cut-1.0/blob/main/Evaluation/retrieval_evaluatoin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install qdrant-client -q
!pip install fastembed-gpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.2/283.2 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.1 MB/s eta 0:00:00


In [3]:
import json
import pandas as pd
from typing import List

from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding

### Load documents and ground truth

In [4]:
with open('Data/data.json', 'rt') as f_in:
    documents = json.load(f_in)

In [7]:
gt_df =  pd.read_csv('ground_truth.csv')

In [6]:
client = QdrantClient(location=":memory:")

In [8]:
model_handle = "jinaai/jina-embeddings-v2-small-en"

In [9]:
def build_collection(name_collection: str, EMBEDDING_DIMENSION: int):

    try:
        if name_collection in client.get_collections().collections:
            print(f"Collection '{name_collection}' already exists.")
            return

        client.create_collection(
            collection_name=name_collection,
            vectors_config=models.VectorParams(
                size=EMBEDDING_DIMENSION,
                distance=models.Distance.COSINE
            )
        )

        print(f"Qdrant collection '{name_collection}' created (dimension: {EMBEDDING_DIMENSION})")

    except Exception as e:
        print(f"Failed to create collection '{name_collection}': {e}")


In [10]:
build_collection('Testing', 512)

Qdrant collection 'Testing' created (dimension: 512)


In [11]:
def populate_collection(name_collection: str, name_model: str, documents: List[dict]):
    points = []

    for record in documents:

        text_to_embed = f"{record['term']}: {record['definition']} {record['extra']}"

        point = models.PointStruct(
            id = record['id'],
            vector= models.Document(text = text_to_embed, model = name_model),
            payload={
                'term': record['term'],
                'about': f"{record['definition']} {record['extra']}"
            }
        )

        points.append(point)

    client.upsert(
        collection_name=name_collection,
        points=points
    )

    print(f"Collection '{name_collection}' filled with {len(points)} records.")

In [13]:
populate_collection('Testing', 'jinaai/jina-embeddings-v2-small-en', documents=documents)

Collection 'Testing' filled with 829 records.


In [17]:
def search(question, name_collection='Testing'):

    results = client.query_points(
        collection_name = name_collection,
        query = models.Document(text = question, model=model_handle),
        limit = 5,
        with_payload=True
    )

    best_matches = []
    for scored_point in results.points:
        best_matches.append(scored_point.id)

    return best_matches

In [34]:
import collections
from typing import List

In [35]:
output = collections.defaultdict(list)

for i in range(len(gt_df)):

    output[gt_df.iloc[i].id].append(search(gt_df.iloc[i].question, 'Testing'))

In [ ]:
output

In [38]:
def score_hit_rate(output):

    hits = 0
    for id, document_ids in output.items():
        for document_id in document_ids:
            for doc_id in document_id:
                if id == doc_id:
                    hits += 1
                    break

    return hits

def score_mrr(ouput):

    score = 0.0

    for id, document_ids in output.items():
        for document_id in document_ids:
            for index, doc_id in enumerate(document_id):
                if id == doc_id:
                    score += (1/(index+1))
                    break

    return score

In [40]:
score_hit_rate(output) / len(gt_df), score_mrr(output) / len(gt_df)

(0.7823884197828709, 0.7128106151990354)